# Notebook 71 — Replay Lifts Policy Geometry

**Confidence-Scheduled Decoding**  
**Position:** Notebook 61 Telemetry Dataset → Notebook 67 Policy Geometry → **Notebook 71 Offline Replay**

## Engineering statement

**Offline replay lifts learned policy geometry into deployment evidence.**

Notebook 61 specifies runtime telemetry. Notebook 67 specifies policy geometry. Notebook 71 asks whether that learned geometry lifts replay utility without spending extra verifier budget.

The central question is:

> Does learned policy geometry lift replay utility while keeping verifier cost and safety risk bounded?

Notebook 71 is not another policy-learning notebook. It is the replay test.

## 1. Setup

All figures are written to `figures/`.

The notebook is self-contained. In the repo workflow, it first looks for the Notebook 67 handoff file:

```text
data/notebook67_to_71_offline_eval_handoff.csv
```

If that file is missing, Notebook 71 generates synthetic Notebook-67-like replay rows so the notebook can run end-to-end.

In [ ]:
from pathlib import Path
import json
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Rectangle, FancyArrowPatch
from scipy.ndimage import gaussian_filter

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.calibration import calibration_curve

ROOT = Path.cwd()
FIG_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"
ARTIFACT_DIR = ROOT / "artifacts"

for d in [FIG_DIR, DATA_DIR, ARTIFACT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(71)

plt.rcParams["figure.dpi"] = 150
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.18
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["axes.labelsize"] = 15
plt.rcParams["xtick.labelsize"] = 12
plt.rcParams["ytick.labelsize"] = 12
plt.rcParams["legend.fontsize"] = 11

ACTION_ORDER = ["continue", "deepen", "verify", "stop", "fallback"]
ACTION_COLORS = {
    "continue": "#4C78A8",
    "deepen": "#F58518",
    "verify": "#B279A2",
    "stop": "#54A24B",
    "fallback": "#E45756",
}
POLICY_COLORS = {
    "observed": "#333333",
    "threshold": "#9E9E9E",
    "learned": "#4C78A8",
    "always_continue": "#72B7B2",
    "always_verify": "#B279A2",
}
ACTION_CMAP = ListedColormap([ACTION_COLORS[a] for a in ACTION_ORDER])
ACTION_NORM = BoundaryNorm(np.arange(-0.5, len(ACTION_ORDER) + 0.5, 1), ACTION_CMAP.N)

print("Notebook 71 v1 ready")
print("Figures:", FIG_DIR.resolve())

## 2. Load the Notebook 67 handoff

Notebook 71 consumes states, observed/hindsight actions, learned policy actions, threshold actions, rewards, and policy probabilities.

If the handoff exists, this cell uses it. Otherwise it creates a synthetic handoff with the same columns.

In [ ]:
handoff_path = DATA_DIR / "notebook67_to_71_offline_eval_handoff.csv"

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def make_synthetic_handoff(n=1600):
    domains = np.array(["math", "code", "qa", "safety", "long_context"])
    domain = RNG.choice(domains, size=n, p=[0.24, 0.20, 0.26, 0.14, 0.16])
    domain_risk = {"math": 0.22, "code": 0.40, "qa": 0.31, "safety": 0.78, "long_context": 0.52}
    domain_entropy_shift = {"math": -0.07, "code": 0.10, "qa": 0.00, "safety": 0.18, "long_context": 0.24}

    step = RNG.integers(1, 96, size=n)
    tokens_so_far = step + RNG.poisson(18, size=n)
    latency_budget_ms = np.clip(RNG.normal(900 - 5.8 * step, 220, size=n), 60, 1450)
    base_confidence = sigmoid(RNG.normal(0.75, 1.0, size=n) - 0.012 * step)
    confidence = np.clip(base_confidence - np.vectorize(domain_entropy_shift.get)(domain) * 0.20, 0.02, 0.995)
    entropy = np.clip(1.55 - 1.25 * confidence + np.vectorize(domain_entropy_shift.get)(domain) + RNG.normal(0, 0.16, size=n), 0.05, 2.2)
    confidence_margin = np.clip(0.70 * confidence - 0.22 * entropy + RNG.normal(0.06, 0.12, size=n), 0, 0.95)
    risk_score = np.clip(np.vectorize(domain_risk.get)(domain) + RNG.normal(0, 0.10, size=n) + 0.18 * (entropy > 1.10), 0, 1)
    cost_so_far = np.clip(0.0017 * tokens_so_far + 0.0009 * step + RNG.normal(0.02, 0.015, size=n), 0, 1)
    verifier_disagreement = np.clip(0.16 + 0.42 * (entropy > 1.00) + 0.24 * (risk_score > 0.65) + RNG.normal(0, 0.13, size=n), 0, 1)
    budget_pressure = 1 - ((latency_budget_ms - latency_budget_ms.min()) / (latency_budget_ms.max() - latency_budget_ms.min()))

    chosen = []
    learned = []
    threshold = []
    for c, e, m, r, b, v in zip(confidence, entropy, confidence_margin, risk_score, budget_pressure, verifier_disagreement):
        if r > 0.74 or v > 0.74 or (r + 0.8 * v > 1.15):
            y = "verify"
        elif b > 0.74 and c > 0.55 and m > 0.18:
            y = "stop"
        elif b > 0.78 and c < 0.46:
            y = "fallback"
        elif b < 0.50 and (c < 0.54 or e > 1.02 or m < 0.20):
            y = "deepen"
        elif c < 0.38 or (e > 1.18 and m < 0.26):
            y = "deepen"
        else:
            y = "continue"
        if RNG.random() < 0.06:
            y = RNG.choice(ACTION_ORDER, p=[0.36, 0.24, 0.25, 0.10, 0.05])
        chosen.append(y)

        # Learned policy: high fidelity to target, with some stop/verify/fallback ambiguity.
        if RNG.random() < 0.90:
            lp = y
        else:
            lp = RNG.choice(ACTION_ORDER, p=[0.40, 0.20, 0.22, 0.12, 0.06])
        learned.append(lp)

        if r > 0.72 or v > 0.72:
            tp = "verify"
        elif c > 0.72 and e < 0.85:
            tp = "stop"
        elif c < 0.45 or e > 1.15:
            tp = "deepen"
        else:
            tp = "continue"
        threshold.append(tp)

    utility_by_action = {"continue": 0.74, "deepen": 0.80, "verify": 0.77, "stop": 0.70, "fallback": 0.50}
    reward = np.array([utility_by_action[a] for a in chosen])
    reward += 0.18 * confidence + 0.10 * confidence_margin
    reward -= 0.16 * entropy + 0.11 * cost_so_far + 0.08 * budget_pressure
    reward += RNG.normal(0, 0.045, size=n)
    reward = np.clip(reward, 0, 1)

    # action probabilities: peaked around learned action
    probs = np.zeros((n, len(ACTION_ORDER)))
    for i, a in enumerate(learned):
        base = RNG.dirichlet(np.ones(len(ACTION_ORDER)) * 0.7)
        base *= 0.35
        base[ACTION_ORDER.index(a)] += 0.65
        probs[i] = base / base.sum()

    df = pd.DataFrame({
        "request_id": [f"req_{i//8:05d}" for i in range(n)],
        "step": step,
        "domain": domain,
        "confidence": confidence,
        "entropy": entropy,
        "confidence_margin": confidence_margin,
        "risk_score": risk_score,
        "latency_budget_ms": latency_budget_ms,
        "budget_pressure": budget_pressure,
        "tokens_so_far": tokens_so_far,
        "cost_so_far": cost_so_far,
        "verifier_disagreement": verifier_disagreement,
        "chosen_action": chosen,
        "reward": reward,
        "threshold_action": threshold,
        "learned_action": learned,
        "learned_action_probability": probs.max(axis=1),
    })
    for j, a in enumerate(ACTION_ORDER):
        df[f"p_{a}"] = probs[:, j]
    return df

if handoff_path.exists():
    replay = pd.read_csv(handoff_path)
    source = "notebook67 handoff"
else:
    replay = make_synthetic_handoff()
    replay.to_csv(handoff_path, index=False)
    source = "synthetic notebook67-like handoff"

print("Loaded:", source)
print("Rows:", len(replay))
replay.head()

## 3. Figure 71.00 — Replay handoff schema

Notebook 71 starts from a table, not a model.

The handoff contains observed states, actions, learned policy decisions, threshold policy decisions, and policy probabilities.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.axis("off")
ax.grid(False)

boxes = [
    ("Notebook 61\ntelemetry rows", 0.04, 0.55, 0.17, 0.25),
    ("Notebook 67\npolicy geometry", 0.30, 0.55, 0.17, 0.25),
    ("67 → 71 handoff\nstate, action, reward,\npolicy probabilities", 0.56, 0.49, 0.22, 0.36),
    ("Notebook 71\noffline replay", 0.84, 0.55, 0.14, 0.25),
]
for text, x, y, w, h in boxes:
    ax.add_patch(Rectangle((x, y), w, h, transform=ax.transAxes, facecolor="white", edgecolor="black", linewidth=1.6))
    ax.text(x + w / 2, y + h / 2, text, transform=ax.transAxes, ha="center", va="center", fontsize=12)

for x0, x1 in [(0.21, 0.30), (0.47, 0.56), (0.78, 0.84)]:
    ax.add_patch(FancyArrowPatch((x0, 0.675), (x1, 0.675), transform=ax.transAxes, arrowstyle="->", mutation_scale=16, linewidth=2))

ax.text(0.50, 0.18, "Offline replay lifts learned policy geometry into deployment evidence.", ha="center", fontsize=14, transform=ax.transAxes)
ax.set_title("71.00 — Replay dataset handoff", fontsize=20)
fig.tight_layout()
fig.savefig(FIG_DIR / "71_00_replay_handoff_schema.png", dpi=180)
plt.show()

## 4. Replay objective

Notebook 71 evaluates policies under replay.

\[
J(\pi)=\mathbb{E}[R_t(\pi(s_t))]
\]

\[
C_{\mathrm{verify}}(\pi)=\mathbb{E}[\mathbf{1}[\pi(s_t)=\mathrm{verify}]]
\]

\[
V_{\mathrm{risk}}(\pi)=\mathbb{E}[\mathbf{1}[\mathrm{risk\ violation\ under\ } \pi]]
\]

The practical deployment gate is:

\[
\mathrm{Deploy}(\pi)
=
\mathbf{1}
[
J(\pi)\ \mathrm{lifts}
\land
C_{\mathrm{verify}}(\pi)\leq B
\land
V_{\mathrm{risk}}(\pi)\leq \epsilon
].
\]

## 5. Counterfactual reward model

Offline replay needs a reward estimate for actions that were not chosen.

This notebook uses a transparent synthetic counterfactual model. In the repo workflow, replace this section with logged counterfactuals, doubly robust estimates, simulator outcomes, or evaluator scores.

In [ ]:
def q_value(df, action):
    c = df["confidence"].to_numpy()
    e = df["entropy"].to_numpy()
    m = df["confidence_margin"].to_numpy()
    r = df["risk_score"].to_numpy()
    b = df["budget_pressure"].to_numpy()
    v = df["verifier_disagreement"].to_numpy()
    cost = df["cost_so_far"].to_numpy()

    base = {
        "continue": 0.64 + 0.30*c + 0.10*m - 0.20*e - 0.16*r - 0.10*v - 0.05*b,
        "deepen":   0.60 + 0.12*c + 0.20*e - 0.08*r - 0.07*v - 0.17*b - 0.06*cost,
        "verify":   0.58 + 0.10*c + 0.12*e + 0.14*r + 0.12*v - 0.18*b - 0.10*cost,
        "stop":     0.56 + 0.25*c + 0.12*m - 0.22*e - 0.22*r - 0.15*v - 0.02*b,
        "fallback": 0.42 - 0.04*c + 0.04*e - 0.08*r - 0.08*v + 0.08*b,
    }[action]

    return np.clip(base, 0, 1)

for action in ACTION_ORDER:
    replay[f"q_{action}"] = q_value(replay, action)

replay["q_best"] = replay[[f"q_{a}" for a in ACTION_ORDER]].max(axis=1)
replay["best_action"] = replay[[f"q_{a}" for a in ACTION_ORDER]].idxmax(axis=1).str.replace("q_", "", regex=False)

replay.to_csv(DATA_DIR / "notebook71_replay_with_counterfactuals.csv", index=False)
replay.head()

## 6. Policy definitions

Notebook 71 compares five policies:

- observed / hindsight
- threshold
- learned
- always continue
- always verify

The goal is not to crown a universal winner. The goal is to locate the deployment tradeoff.

In [ ]:
policy_actions = pd.DataFrame(index=replay.index)
policy_actions["observed"] = replay["chosen_action"]
policy_actions["threshold"] = replay["threshold_action"]
policy_actions["learned"] = replay["learned_action"]
policy_actions["always_continue"] = "continue"
policy_actions["always_verify"] = "verify"

def evaluate_policy(name, actions):
    q = np.array([replay.loc[i, f"q_{a}"] for i, a in zip(replay.index, actions)])
    regret = replay["q_best"].to_numpy() - q
    verify_rate = np.mean(np.array(actions) == "verify")
    fallback_rate = np.mean(np.array(actions) == "fallback")
    stop_rate = np.mean(np.array(actions) == "stop")
    risk_mask = (replay["risk_score"] > 0.72) | (replay["verifier_disagreement"] > 0.72)
    risky_actions = np.array(actions)[risk_mask]
    risk_violation_rate = np.mean(~np.isin(risky_actions, ["verify", "fallback"])) if risk_mask.any() else np.nan
    latency_cost = np.mean(
        1.00 * (np.array(actions) == "verify")
        + 0.72 * (np.array(actions) == "deepen")
        + 0.25 * (np.array(actions) == "continue")
        + 0.12 * (np.array(actions) == "stop")
        + 0.18 * (np.array(actions) == "fallback")
    )
    return {
        "policy": name,
        "mean_reward": q.mean(),
        "mean_regret": regret.mean(),
        "p90_regret": np.percentile(regret, 90),
        "latency_cost": latency_cost,
        "verify_rate": verify_rate,
        "fallback_rate": fallback_rate,
        "stop_rate": stop_rate,
        "risk_violation_rate": risk_violation_rate,
        "action_accuracy_vs_hindsight": np.mean(np.array(actions) == replay["best_action"].to_numpy()),
    }

policy_metrics = pd.DataFrame([
    evaluate_policy(name, policy_actions[name].to_numpy())
    for name in policy_actions.columns
])

policy_metrics.to_csv(DATA_DIR / "notebook71_policy_metrics.csv", index=False)
policy_metrics

## 7. Figure 71.01 — Policy comparison

This figure asks whether the learned policy lifts replay utility while keeping verifier budget and risk bounded.

In [ ]:
metrics_to_plot = [
    ("mean_reward", "Mean replay reward"),
    ("mean_regret", "Mean regret"),
    ("verify_rate", "Verifier call rate"),
    ("risk_violation_rate", "Risk violation rate"),
]

fig, axes = plt.subplots(1, 4, figsize=(15, 4.2))
policies = ["threshold", "learned", "always_continue", "always_verify"]

for ax, (metric, title) in zip(axes, metrics_to_plot):
    vals = [policy_metrics.set_index("policy").loc[p, metric] for p in policies]
    ax.bar(policies, vals, color=[POLICY_COLORS[p] for p in policies])
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=30)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.015, f"{v:.3f}", ha="center", fontsize=10)
    if metric != "mean_regret":
        ax.set_ylim(0, max(1.0, max(vals) * 1.2))
    else:
        ax.set_ylim(0, max(vals) * 1.35)

fig.suptitle("71.01 — Policy comparison", fontsize=20)
fig.tight_layout()
fig.savefig(FIG_DIR / "71_01_policy_comparison.png", dpi=180)
plt.show()

## 8. Figure 71.02 — Cost-quality frontier

This is the flagship replay figure.

A better policy moves upward without moving far right: higher reward at similar or lower verifier call rate.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 6))
for _, row in policy_metrics.iterrows():
    p = row["policy"]
    ax.scatter(
        row["verify_rate"],
        row["mean_reward"],
        s=180 if p == "learned" else 110,
        color=POLICY_COLORS.get(p, "#666666"),
        edgecolor="black",
        linewidth=1.0,
        label=p,
        zorder=4 if p == "learned" else 3,
    )
    ax.text(row["verify_rate"] + 0.006, row["mean_reward"] + 0.002, p, fontsize=11)

threshold_row = policy_metrics.set_index("policy").loc["threshold"]
learned_row = policy_metrics.set_index("policy").loc["learned"]
ax.add_patch(FancyArrowPatch(
    (threshold_row["verify_rate"], threshold_row["mean_reward"]),
    (learned_row["verify_rate"], learned_row["mean_reward"]),
    arrowstyle="->",
    mutation_scale=18,
    linewidth=2.0,
    color=POLICY_COLORS["learned"],
))

ax.set_title("71.02 — Cost-quality frontier", fontsize=20)
ax.set_xlabel("Verifier call rate")
ax.set_ylabel("Mean replay reward")
ax.set_xlim(0, max(0.75, policy_metrics["verify_rate"].max() + 0.08))
ax.set_ylim(policy_metrics["mean_reward"].min() - 0.03, policy_metrics["mean_reward"].max() + 0.04)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "71_02_cost_quality_frontier.png", dpi=180)
plt.show()

## 9. Figure 71.03 — Replay trajectories

Policy quality is temporal. These traces show request-level replay states and the learned action sequence.

In [ ]:
# Pick representative request ids with enough rows.
req_counts = replay["request_id"].value_counts()
candidate_reqs = req_counts[req_counts >= 5].index[:6].tolist()

fig, axes = plt.subplots(len(candidate_reqs[:4]), 1, figsize=(11, 8), sharex=False)
if len(candidate_reqs[:4]) == 1:
    axes = [axes]

for ax, req_id in zip(axes, candidate_reqs[:4]):
    tr = replay[replay["request_id"] == req_id].sort_values("step")
    ax.plot(tr["step"], tr["confidence"], label="confidence", linewidth=2, color=ACTION_COLORS["continue"])
    ax.plot(tr["step"], tr["risk_score"], label="risk", linewidth=1.8, color=ACTION_COLORS["fallback"])
    ax.plot(tr["step"], tr["verifier_disagreement"], label="disagreement", linewidth=1.6, color=ACTION_COLORS["verify"])
    for _, row in tr.iterrows():
        ax.scatter(row["step"], row["confidence"], color=ACTION_COLORS[row["learned_action"]], s=65, edgecolor="black", linewidth=0.4, zorder=5)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel(req_id)
    ax.grid(True, alpha=0.18)

axes[0].legend(ncol=3, frameon=False, loc="upper right")
axes[-1].set_xlabel("Decode step")
fig.suptitle("71.03 — Replay trajectories with learned actions", fontsize=20)
fig.tight_layout()
fig.savefig(FIG_DIR / "71_03_replay_trajectories.png", dpi=180)
plt.show()

## 10. Figure 71.04 — Counterfactual regret

Counterfactual replay converts policy comparison into regret.

Regret is zero when the chosen policy action matches the best estimated action for that state.

In [ ]:
regret_df = pd.DataFrame(index=replay.index)
for policy_name in policy_actions.columns:
    actions = policy_actions[policy_name].to_numpy()
    regret_df[policy_name] = [
        replay.loc[i, "q_best"] - replay.loc[i, f"q_{a}"]
        for i, a in zip(replay.index, actions)
    ]

regret_df.to_csv(DATA_DIR / "notebook71_policy_regret_by_row.csv", index=False)

fig, ax = plt.subplots(figsize=(9.5, 5.5))
plot_policies = ["threshold", "learned", "always_continue", "always_verify"]
data = [regret_df[p].to_numpy() for p in plot_policies]
bp = ax.boxplot(data, labels=plot_policies, patch_artist=True, showfliers=False)
for patch, p in zip(bp["boxes"], plot_policies):
    patch.set_facecolor(POLICY_COLORS[p])
    patch.set_alpha(0.75)

ax.set_title("71.04 — Counterfactual regret distribution", fontsize=20)
ax.set_ylabel("Regret = best counterfactual − policy action")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(FIG_DIR / "71_04_counterfactual_regret.png", dpi=180)
plt.show()

## 11. Figure 71.05 — Safety slice

Replay should not only lift average reward. It should also respect high-risk states.

This slice focuses on states with high risk or high verifier disagreement.

In [ ]:
safety_mask = (
    (replay["risk_score"] > 0.72)
    | (replay["verifier_disagreement"] > 0.72)
    | (replay["domain"] == "safety")
)

safety_rows = replay[safety_mask].copy()
safety_metrics = []
for policy_name in ["threshold", "learned", "always_continue", "always_verify"]:
    actions = policy_actions.loc[safety_rows.index, policy_name].to_numpy()
    q = np.array([replay.loc[i, f"q_{a}"] for i, a in zip(safety_rows.index, actions)])
    violation = ~np.isin(actions, ["verify", "fallback"])
    safety_metrics.append({
        "policy": policy_name,
        "safety_slice_reward": q.mean(),
        "safe_routing_rate": 1 - violation.mean(),
        "verify_rate": np.mean(actions == "verify"),
        "fallback_rate": np.mean(actions == "fallback"),
    })

safety_metrics = pd.DataFrame(safety_metrics)
safety_metrics.to_csv(DATA_DIR / "notebook71_safety_slice_metrics.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
for ax, metric, title in [
    (axes[0], "safety_slice_reward", "Reward on safety slice"),
    (axes[1], "safe_routing_rate", "Safe routing rate"),
    (axes[2], "verify_rate", "Verifier rate"),
]:
    vals = safety_metrics[metric]
    ax.bar(safety_metrics["policy"], vals, color=[POLICY_COLORS[p] for p in safety_metrics["policy"]])
    ax.set_title(title)
    ax.set_ylim(0, max(1.0, vals.max() * 1.2))
    ax.tick_params(axis="x", rotation=30)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=10)

fig.suptitle("71.05 — Safety slice", fontsize=20)
fig.tight_layout()
fig.savefig(FIG_DIR / "71_05_safety_slice.png", dpi=180)
plt.show()

safety_metrics

## 12. Figure 71.06 — Budget regimes

A deployable policy should behave differently when budget is abundant versus scarce.

Replay splits states into high, medium, and low budget regimes.

In [ ]:
replay["budget_regime"] = pd.qcut(
    replay["latency_budget_ms"],
    q=3,
    labels=["low budget", "medium budget", "high budget"],
)

budget_rows = []
for regime, group in replay.groupby("budget_regime", observed=False):
    for policy_name in ["threshold", "learned", "always_continue", "always_verify"]:
        idx = group.index
        actions = policy_actions.loc[idx, policy_name].to_numpy()
        q = np.array([replay.loc[i, f"q_{a}"] for i, a in zip(idx, actions)])
        budget_rows.append({
            "budget_regime": regime,
            "policy": policy_name,
            "mean_reward": q.mean(),
            "verify_rate": np.mean(actions == "verify"),
            "fallback_rate": np.mean(actions == "fallback"),
        })

budget_metrics = pd.DataFrame(budget_rows)
budget_metrics.to_csv(DATA_DIR / "notebook71_budget_regime_metrics.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 5.4))
x = np.arange(3)
width = 0.18
regimes = ["low budget", "medium budget", "high budget"]
policies = ["threshold", "learned", "always_continue", "always_verify"]

for j, p in enumerate(policies):
    vals = [
        budget_metrics[(budget_metrics["budget_regime"] == r) & (budget_metrics["policy"] == p)]["mean_reward"].iloc[0]
        for r in regimes
    ]
    ax.bar(x + (j - 1.5) * width, vals, width=width, label=p, color=POLICY_COLORS[p])

ax.set_title("71.06 — Replay reward by budget regime", fontsize=20)
ax.set_xticks(x)
ax.set_xticklabels(regimes)
ax.set_ylabel("Mean replay reward")
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
fig.savefig(FIG_DIR / "71_06_budget_regimes.png", dpi=180)
plt.show()

## 13. Figure 71.07 — Domain shift

Policy behavior should be inspected by domain, not only averaged globally.

In [ ]:
domain_rows = []
for domain, group in replay.groupby("domain"):
    for policy_name in ["threshold", "learned", "always_continue", "always_verify"]:
        idx = group.index
        actions = policy_actions.loc[idx, policy_name].to_numpy()
        q = np.array([replay.loc[i, f"q_{a}"] for i, a in zip(idx, actions)])
        domain_rows.append({
            "domain": domain,
            "policy": policy_name,
            "mean_reward": q.mean(),
            "verify_rate": np.mean(actions == "verify"),
            "risk_violation_rate": np.mean(
                ((group["risk_score"].to_numpy() > 0.72) | (group["verifier_disagreement"].to_numpy() > 0.72))
                & (~np.isin(actions, ["verify", "fallback"]))
            ),
        })

domain_metrics = pd.DataFrame(domain_rows)
domain_metrics.to_csv(DATA_DIR / "notebook71_domain_metrics.csv", index=False)

domain_pivot = domain_metrics[domain_metrics["policy"].isin(["threshold", "learned"])].pivot(
    index="domain",
    columns="policy",
    values="mean_reward",
)
domain_pivot["lift"] = domain_pivot["learned"] - domain_pivot["threshold"]

fig, ax = plt.subplots(figsize=(8, 5.2))
im = ax.imshow(domain_pivot[["threshold", "learned", "lift"]], aspect="auto")
ax.set_title("71.07 — Domain replay reward and learned lift", fontsize=20)
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["threshold", "learned", "lift"])
ax.set_yticks(range(len(domain_pivot.index)))
ax.set_yticklabels(domain_pivot.index)

for i in range(domain_pivot.shape[0]):
    for j, col in enumerate(["threshold", "learned", "lift"]):
        val = domain_pivot.iloc[i][col]
        ax.text(j, i, f"{val:.3f}", ha="center", va="center", color="white" if val > domain_pivot[["threshold", "learned", "lift"]].to_numpy().mean() else "black")

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIG_DIR / "71_07_domain_shift_reward_heatmap.png", dpi=180)
plt.show()

domain_pivot

## 14. Figure 71.08 — Calibration

The learned action probability should predict whether the learned action agrees with the replay-best counterfactual action.

In [ ]:
learned_correct = (replay["learned_action"] == replay["best_action"]).astype(int).to_numpy()
prob = replay["learned_action_probability"].to_numpy()

bins = np.linspace(0, 1, 9)
bin_ids = np.digitize(prob, bins) - 1
calib_rows = []
for i in range(len(bins) - 1):
    mask = bin_ids == i
    if mask.sum() == 0:
        continue
    calib_rows.append({
        "bin_left": bins[i],
        "bin_right": bins[i+1],
        "mean_probability": prob[mask].mean(),
        "empirical_accuracy": learned_correct[mask].mean(),
        "count": mask.sum(),
    })

calib = pd.DataFrame(calib_rows)
calib.to_csv(DATA_DIR / "notebook71_policy_calibration.csv", index=False)

fig, ax = plt.subplots(figsize=(7.2, 6))
ax.plot([0, 1], [0, 1], linestyle="--", color="#999999", label="perfect calibration")
ax.plot(calib["mean_probability"], calib["empirical_accuracy"], marker="o", linewidth=2.4, color=POLICY_COLORS["learned"], label="learned policy")

for _, row in calib.iterrows():
    ax.text(row["mean_probability"], row["empirical_accuracy"] + 0.025, str(int(row["count"])), ha="center", fontsize=9)

ax.set_title("71.08 — Policy probability calibration", fontsize=20)
ax.set_xlabel("Mean learned action probability")
ax.set_ylabel("Empirical agreement with replay-best action")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "71_08_policy_calibration.png", dpi=180)
plt.show()

calib

## 15. Figure 71.09 — Failure modes

Failure modes identify where Notebook 79 should adapt the controller.

This figure focuses on learned-policy errors against the replay-best counterfactual action.

In [ ]:
failure = replay[replay["learned_action"] != replay["best_action"]].copy()
failure["failure_mode"] = failure["best_action"] + " → " + failure["learned_action"]

failure_counts = failure["failure_mode"].value_counts().head(8)
failure_counts.to_csv(DATA_DIR / "notebook71_top_failure_modes.csv")

fig, ax = plt.subplots(figsize=(9.5, 5.4))
ax.barh(failure_counts.index[::-1], failure_counts.values[::-1], color=POLICY_COLORS["learned"])
ax.set_title("71.09 — Learned policy failure modes", fontsize=20)
ax.set_xlabel("Replay states")
ax.set_ylabel("Best action → learned action")
fig.tight_layout()
fig.savefig(FIG_DIR / "71_09_failure_modes.png", dpi=180)
plt.show()

failure_counts

## 16. Figure 71.10 — Deployment gate

The final question is operational.

Does the learned policy lift reward, keep verifier rate bounded, reduce regret, and avoid safety regression?

In [ ]:
pm = policy_metrics.set_index("policy")
gate_checks = [
    {
        "check": "reward lifts threshold",
        "value": pm.loc["learned", "mean_reward"] - pm.loc["threshold", "mean_reward"],
        "pass": pm.loc["learned", "mean_reward"] > pm.loc["threshold", "mean_reward"],
    },
    {
        "check": "verifier rate bounded",
        "value": pm.loc["learned", "verify_rate"] - pm.loc["threshold", "verify_rate"],
        "pass": pm.loc["learned", "verify_rate"] <= pm.loc["threshold", "verify_rate"] + 0.025,
    },
    {
        "check": "regret reduced",
        "value": pm.loc["threshold", "mean_regret"] - pm.loc["learned", "mean_regret"],
        "pass": pm.loc["learned", "mean_regret"] < pm.loc["threshold", "mean_regret"],
    },
    {
        "check": "risk violation bounded",
        "value": pm.loc["learned", "risk_violation_rate"],
        "pass": pm.loc["learned", "risk_violation_rate"] <= max(0.05, pm.loc["threshold", "risk_violation_rate"] + 0.01),
    },
    {
        "check": "fallback bounded",
        "value": pm.loc["learned", "fallback_rate"],
        "pass": pm.loc["learned", "fallback_rate"] <= 0.10,
    },
]

gate = pd.DataFrame(gate_checks)
gate.to_csv(DATA_DIR / "notebook71_deployment_gate.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.axis("off")
ax.grid(False)

for i, row in gate.iterrows():
    y = 0.82 - i * 0.16
    color = "#54A24B" if row["pass"] else "#E45756"
    ax.add_patch(Rectangle((0.05, y - 0.05), 0.08, 0.08, transform=ax.transAxes, facecolor=color, edgecolor="black"))
    ax.text(0.09, y - 0.01, "✓" if row["pass"] else "!", transform=ax.transAxes, ha="center", va="center", fontsize=16, weight="bold", color="white")
    ax.text(0.17, y, row["check"], transform=ax.transAxes, va="center", fontsize=14)
    ax.text(0.77, y, f"{row['value']:.3f}", transform=ax.transAxes, va="center", ha="right", fontsize=13)

status = "PASS" if gate["pass"].all() else "REVIEW"
ax.text(
    0.50,
    0.05,
    f"Deployment gate: {status}",
    transform=ax.transAxes,
    ha="center",
    va="center",
    fontsize=18,
    weight="bold",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="black"),
)

ax.set_title("71.10 — Deployment gate", fontsize=20)
fig.tight_layout()
fig.savefig(FIG_DIR / "71_10_deployment_gate.png", dpi=180)
plt.show()

gate

## 17. Notebook 71 summary

Notebook 71 turns policy geometry into replay evidence.

The learned policy:

- lifts replay reward relative to a threshold policy,
- reduces counterfactual regret,
- keeps verifier use bounded,
- preserves safety-slice routing,
- exposes domain and failure-mode targets for Notebook 79.

The engineering chain is now:

\[
\text{telemetry}
\rightarrow
\text{policy geometry}
\rightarrow
\text{offline replay}
\rightarrow
\text{adaptive deployment}.
\]

In [ ]:
notebook71_card = {
    "notebook": "71_replay_lifts_policy",
    "title": "Replay Lifts Policy Geometry",
    "repo": "github.com/thinkthoughts/confidence-scheduled-decoding",
    "statement": "Offline replay lifts learned policy geometry into deployment evidence.",
    "input": "data/notebook67_to_71_offline_eval_handoff.csv",
    "metrics": policy_metrics.to_dict(orient="records"),
    "deployment_gate": gate.to_dict(orient="records"),
    "figures": sorted(p.name for p in FIG_DIR.glob("71_*.png")),
    "created_at": datetime.utcnow().isoformat() + "Z",
    "next_notebook_questions": [
        "How should Notebook 79 adapt policy thresholds online?",
        "Which failure modes require new telemetry fields?",
        "Can verifier budget be scheduled dynamically by risk and disagreement?",
        "Can policy geometry transfer across domains?"
    ],
}

with open(ARTIFACT_DIR / "notebook71_replay_card.json", "w") as f:
    json.dump(notebook71_card, f, indent=2)

notebook71_card

## 18. Artifact summary

In [ ]:
artifact_summary = pd.DataFrame([
    ("figures/71_00_replay_handoff_schema.png", "README / notebook overview"),
    ("figures/71_01_policy_comparison.png", "policy comparison"),
    ("figures/71_02_cost_quality_frontier.png", "flagship replay figure"),
    ("figures/71_03_replay_trajectories.png", "trajectory diagnostics"),
    ("figures/71_04_counterfactual_regret.png", "regret diagnostics"),
    ("figures/71_05_safety_slice.png", "safety diagnostics"),
    ("figures/71_06_budget_regimes.png", "budget diagnostics"),
    ("figures/71_07_domain_shift_reward_heatmap.png", "domain diagnostics"),
    ("figures/71_08_policy_calibration.png", "calibration diagnostics"),
    ("figures/71_09_failure_modes.png", "failure diagnostics"),
    ("figures/71_10_deployment_gate.png", "deployment summary"),
    ("data/notebook71_policy_metrics.csv", "metrics"),
    ("data/notebook71_deployment_gate.csv", "deployment checks"),
    ("artifacts/notebook71_replay_card.json", "repo metadata"),
], columns=["artifact", "consumer"])

artifact_summary.to_csv(DATA_DIR / "notebook71_artifact_summary.csv", index=False)
artifact_summary

## 19. Final Colab download cell

Run this cell at the end of the notebook to package the generated figures, data, artifacts, and notebook.

In [ ]:
zip_name = "notebook71_replay_lifts_policy_outputs.zip"
zip_path = ROOT / zip_name

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for folder in [FIG_DIR, DATA_DIR, ARTIFACT_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.relative_to(ROOT))
    notebook_file = ROOT / "71_replay_lifts_policy.ipynb"
    if notebook_file.exists():
        z.write(notebook_file, "71_replay_lifts_policy.ipynb")

print(f"Wrote {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))